# Exploración inicial

Celda 1: cargar los datos

In [1]:
from pathlib import Path
import pandas as pd

BASE        = Path(r"C:\Users\fsociety\Videos\Proyecto_EEVV")
PROCESADO   = BASE / "datos" / "procesado"
DICCIONARIO = BASE / "documentacion" / "diccionario_variables_eevv.xlsx"

defun = pd.read_parquet(PROCESADO / "defunciones_primera_infancia_pacifico.parquet")
nac   = pd.read_parquet(PROCESADO / "nacimientos_pacifico.parquet")
dic   = pd.read_excel(DICCIONARIO, sheet_name="variables", dtype=str)

defun["grupo_edad_analisis"] = defun["grupo_edad_analisis"].cat.reorder_categories(
    ["Neonatal", "Posneonatal", "1 a 4 años"], ordered=True)

print("Defunciones:", defun.shape, "| Nacimientos:", nac.shape, "| Diccionario:", dic.shape)

Defunciones: (3217, 67) | Nacimientos: (211367, 44) | Diccionario: (106, 13)


Celda 2: funciones de apoyo

In [2]:
def etiquetas(columna, grupo="defunciones_no_fetales"):
    fila = dic[(dic["grupo"] == grupo) & (dic["nombre_nuevo"] == columna)]
    texto = fila["categorias"].iloc[0] if len(fila) else ""
    mapa = {}
    for parte in str(texto).split(";"):
        if "=" in parte:
            codigo, etiqueta = parte.split("=", 1)
            mapa[codigo.strip()] = etiqueta.strip()
    return mapa

def con_etiquetas(serie, columna, grupo="defunciones_no_fetales"):
    return serie.astype("string").replace(etiquetas(columna, grupo)).fillna("Vacío")

def frecuencias(serie):
    t = serie.value_counts(dropna=False).to_frame("n")
    t["%"] = (t["n"] / t["n"].sum() * 100).round(1)
    return t

Celda 3: composición del subconjunto

In [3]:
print(frecuencias(con_etiquetas(defun["sexo"], "sexo")), "\n")
print(frecuencias(defun["grupo_edad_analisis"]), "\n")
print(frecuencias(defun["dpto_residencia_nombre"]), "\n")
print(frecuencias(con_etiquetas(defun["area_residencia"], "area_residencia")))

                  n     %
sexo                     
Masculino      1756  54.6
Femenino       1459  45.4
Indeterminado     2   0.1 

                        n     %
grupo_edad_analisis            
Neonatal             1524  47.4
Posneonatal          1021  31.7
1 a 4 años            672  20.9 

                           n     %
dpto_residencia_nombre            
Valle del Cauca         1323  41.1
Chocó                    789  24.5
Cauca                    584  18.2
Nariño                   521  16.2 

                                                       n     %
area_residencia                                               
Cabecera municipal                                  1879  58.4
Rural disperso                                      1065  33.1
Centro poblado (inspección, corregimiento o cas...   270   8.4
Sin información                                        3   0.1


Celda 4: edad según departamento y sexo

In [4]:
print(pd.crosstab(defun["dpto_residencia_nombre"], defun["grupo_edad_analisis"],
                  normalize="index").mul(100).round(1), "\n")
print(pd.crosstab(con_etiquetas(defun["sexo"], "sexo"), defun["grupo_edad_analisis"],
                  margins=True, margins_name="Total"))

grupo_edad_analisis     Neonatal  Posneonatal  1 a 4 años
dpto_residencia_nombre                                   
Cauca                       49.3         30.1        20.5
Chocó                       31.1         40.3        28.6
Nariño                      46.1         28.6        25.3
Valle del Cauca             56.8         28.6        14.7 

grupo_edad_analisis  Neonatal  Posneonatal  1 a 4 años  Total
sexo                                                         
Femenino                  654          479         326   1459
Indeterminado               2            0           0      2
Masculino                 868          542         346   1756
Total                    1524         1021         672   3217


## Vacíos y coherencia

Celda 5: vacíos por variable

In [5]:
reglas  = dic[dic["grupo"] == "defunciones_no_fetales"].set_index("nombre_nuevo")
claves  = reglas.index[reglas["prioridad_sugerida"].isin(["núcleo", "apoyo"])]
menor_1 = defun["grupo_edad_analisis"] != "1 a 4 años"

def codigos(valor):
    if pd.isna(valor) or "verificar" in valor:
        return []
    return [c.strip() for c in valor.split(",") if c.strip()]

def faltante(s, col):
    return s.isna() | s.astype("string").isin(codigos(reglas.loc[col, "codigos_sin_info"]))

filas = []
for col in claves:
    solo_menor_1 = defun.loc[~menor_1, col].isna().all() and defun.loc[menor_1, col].notna().any()
    s = defun.loc[menor_1, col] if solo_menor_1 else defun[col]
    sin_info = s.astype("string").isin(codigos(reglas.loc[col, "codigos_sin_info"]))
    filas.append({"variable": col,
                  "base": "menores de 1 año" if solo_menor_1 else "todos",
                  "registros": len(s),
                  "% vacío": round(s.isna().mean() * 100, 1),
                  "% sin información": round(sin_info.mean() * 100, 1)})

calidad = pd.DataFrame(filas).set_index("variable")
calidad["% faltante"] = calidad["% vacío"] + calidad["% sin información"]
calidad = calidad.sort_values("% faltante", ascending=False)

print(calidad[calidad["% faltante"] > 0].to_string())
print("\nVariables completas:", (calidad["% faltante"] == 0).sum(), "de", len(calidad))

                                      base  registros  % vacío  % sin información  % faltante
variable                                                                                     
nivel_educativo_madre     menores de 1 año       2545      0.0               27.1        27.1
estado_civil_madre        menores de 1 año       2545      0.0               20.8        20.8
tiempo_gestacion          menores de 1 año       2545      0.0               11.8        11.8
tiempo_gestacion_cie      menores de 1 año       2545      0.0               11.8        11.8
tipo_parto                menores de 1 año       2545      0.0                8.3         8.3
tipo_embarazo             menores de 1 año       2545      0.0                7.3         7.3
edad_madre_grupo          menores de 1 año       2545      0.0                6.7         6.7
hijos_nacidos_vivos       menores de 1 año       2545      6.4                0.0         6.4
peso_nacer                menores de 1 año       2545      0

Celda 6: faltantes por departamento

In [6]:
revisar = calidad.index[calidad["% faltante"] >= 5]
por_dpto = {}
for col in revisar:
    base = defun[menor_1] if calidad.loc[col, "base"] == "menores de 1 año" else defun
    por_dpto[col] = (faltante(base[col], col)
                     .groupby(base["dpto_residencia_nombre"], observed=True)
                     .mean().mul(100).round(1))
print(pd.DataFrame(por_dpto).T.to_string())

dpto_residencia_nombre  Cauca  Chocó  Nariño  Valle del Cauca
nivel_educativo_madre    19.6   31.4    22.1             29.8
estado_civil_madre       13.6   21.0    19.8             24.1
tiempo_gestacion          6.0   29.7     8.2              6.6
tiempo_gestacion_cie      6.0   29.7     8.2              6.6
tipo_parto                7.8    7.3     8.7              8.9
tipo_embarazo             7.5    5.3     8.0              7.9
edad_madre_grupo          7.3    3.9     6.9              7.7
hijos_nacidos_vivos       7.1    3.6     7.2              7.3
peso_nacer                3.0    9.9     5.9              4.4


Celda 7: chequeos de coherencia

In [7]:
uno_a_cuatro = ~menor_1
chequeos = {
    "Edad fuera de 00-08":                   (~defun["grupo_edad_detallado"].isin([f"{i:02d}" for i in range(9)])).sum(),
    "Departamento fuera de la región":       (~defun["dpto_residencia_cod"].isin(["27", "76", "19", "52"])).sum(),
    "Año fuera de 2022-2024":                (~defun["anio"].between(2022, 2024)).sum(),
    "1 a 4 años con datos del parto":        defun.loc[uno_a_cuatro, "peso_nacer"].notna().sum(),
    "Causa básica vacía":                    defun["causa_basica_cie10"].isna().sum(),
    "Causa 6/67 con código no válido":       (~defun["causa_lista_667"].str.fullmatch(r"\d{3}")).sum(),
    "Municipios sin nacimientos (sin tasa)": len(set(defun["mun_residencia_divipola"]) - set(nac["mun_residencia_divipola"])),
    "Sexo indeterminado":                    (defun["sexo"] == "3").sum(),
    "Duplicados exactos":                    int(defun["es_duplicado_exacto"].sum()),
    "1 a 4 años con causa perinatal (P)":    defun.loc[uno_a_cuatro, "causa_basica_cie10"].str.startswith("P").sum(),
    "Murieron en otro departamento":         (defun["dpto_ocurrencia_cod"] != defun["dpto_residencia_cod"]).sum(),
}
print(pd.Series(chequeos, name="registros").to_string())

otro_dpto = defun["dpto_ocurrencia_cod"] != defun["dpto_residencia_cod"]
print("\n% que murió fuera de su departamento:")
print(otro_dpto.groupby(defun["dpto_residencia_nombre"], observed=True).mean().mul(100).round(1).to_string())

Edad fuera de 00-08                        0
Departamento fuera de la región            0
Año fuera de 2022-2024                     0
1 a 4 años con datos del parto             0
Causa básica vacía                         0
Causa 6/67 con código no válido            0
Municipios sin nacimientos (sin tasa)      0
Sexo indeterminado                         2
Duplicados exactos                         1
1 a 4 años con causa perinatal (P)        15
Murieron en otro departamento            558

% que murió fuera de su departamento:
dpto_residencia_nombre
Cauca              34.2
Chocó              31.7
Nariño              8.3
Valle del Cauca     4.9
